In [41]:
import pandas as pd
from datetime import timedelta
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import pointbiserialr, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", 100)

In [42]:
# Load cleaned flood dataset
flood_df = pd.read_parquet("../../flood_data/Bhutan_Historical_Floods_1979_2025_cleaned.parquet")

# Load ERA5 dataset
era_df = pd.read_parquet("../../data/merged_era5data/merged_era5_6hour_1979_2025.parquet")

# Quick check
print("Flood data:", flood_df.shape)
print("ERA5 data:", era_df.shape)

Flood data: (113, 14)
ERA5 data: (9183645, 16)


In [43]:
flood_df.head()

,Date,District,Location,Latitude,Longitude,EventType,Description,Rainfall_Time,Gewog,River_Basin,Source,Year,Month,Day
0,1980-07-10,Wangduephodrang,Wangdue town,27.4867,89.8995,Flash Flood,Flash Flood,None,None,None,1,1980,7,10
1,1990-08-08,Sarpang,Sarpang,26.8627,90.2717,Flash Flood,Flash Flood,None,None,None,1,1990,8,8
2,1994-07-10,Punakha,Punakha,27.5916,89.8774,Flood,Monsoon Flood,None,None,None,1,1994,7,10
3,1996-08-14,Phuentsholing,Phuentsholing,26.8635,89.3883,Flood,Major Flood,None,None,None,1,1996,8,14
4,2000-07-16,Phuentsholing,Phuentsholing,26.8635,89.3883,Flood,Urban Flooding,None,None,None,1,2000,7,16


In [44]:
era_df.head()

,latitude,longitude,datetime,temperature,dewpoint,wind_u,wind_v,potential_evaporation,runoff,snow_depth,snowmelt,soil_temperature,sub_surface_runoff,surface_runoff,solar_radiation,precipitation
0,26.5,88.5,1979-01-01 05:30:00+05:30,8.779205,280.687500,1.184982,-0.662811,1.918059e-06,0.000015,0.0,0.0,285.450195,0.000015,0.0,128.0,0.0
1,26.5,88.5,1979-01-01 11:30:00+05:30,23.847076,282.582336,-0.500717,-0.115112,-5.577810e-04,0.000016,0.0,0.0,294.535645,0.000016,0.0,2433472.0,0.0
2,26.5,88.5,1979-01-01 17:30:00+05:30,20.769440,282.469971,0.937103,-0.837021,-6.991206e-06,0.000016,0.0,0.0,293.682373,0.000016,0.0,0.0,0.0
3,26.5,88.5,1979-01-01 23:30:00+05:30,11.567841,281.223572,0.596100,-1.193222,-2.800487e-06,0.000016,0.0,0.0,287.762207,0.000016,0.0,0.0,0.0
4,26.5,88.5,1979-01-02 05:30:00+05:30,9.451782,281.199463,0.905502,-1.021103,3.932510e-07,0.000016,0.0,0.0,285.927246,0.000016,0.0,64.0,0.0


In [45]:
era_df.dtypes

latitude                                      float64
longitude                                     float64
datetime                 datetime64[ns, Asia/Thimphu]
temperature                                   float64
dewpoint                                      float64
wind_u                                        float64
wind_v                                        float64
potential_evaporation                         float64
runoff                                        float64
snow_depth                                    float64
snowmelt                                      float64
soil_temperature                              float64
sub_surface_runoff                            float64
surface_runoff                                float64
solar_radiation                               float64
precipitation                                 float64
dtype: object

In [46]:
flood_df.dtypes

Date             datetime64[ns]
District                 object
Location                 object
Latitude                float64
Longitude               float64
EventType                object
Description              object
Rainfall_Time            object
Gewog                    object
River_Basin              object
Source                    int64
Year                      int32
Month                     int32
Day                       int32
dtype: object

In [47]:
# strip tz info from ERA5 datetimes, keep local wall clock
era_df['datetime'] = era_df['datetime'].dt.tz_localize(None)

# now both are tz-naive and directly comparable
print(era_df['datetime'].dtype)   # datetime64[ns]
print(flood_df['Date'].dtype)     # datetime64[ns]


datetime64[ns]
datetime64[ns]


In [48]:
# Work on copies (assuming flood_df and era_df are already loaded)
flood = flood_df.copy()
era = era_df.copy()

flood.shape, era.shape

((113, 14), (9183645, 16))

In [49]:
# Convenience daily dates
era['date'] = era['datetime'].dt.floor('D')
flood['date'] = flood['Date'].dt.floor('D')

era[['datetime','date']].head(), flood[['Date','date']].head()

(             datetime       date
 0 1979-01-01 05:30:00 1979-01-01
 1 1979-01-01 11:30:00 1979-01-01
 2 1979-01-01 17:30:00 1979-01-01
 3 1979-01-01 23:30:00 1979-01-01
 4 1979-01-02 05:30:00 1979-01-02,
         Date       date
 0 1980-07-10 1980-07-10
 1 1990-08-08 1990-08-08
 2 1994-07-10 1994-07-10
 3 1996-08-14 1996-08-14
 4 2000-07-16 2000-07-16)

## Daily ERA5 Aggregation

In [50]:

# Wind speed
era['wind_speed'] = np.sqrt(era['wind_u']**2 + era['wind_v']**2)

# Aggregate per grid cell per day
agg = {
    'precipitation': ['sum', 'max'],
    'surface_runoff': ['sum', 'mean'],
    'temperature': ['mean', 'max'],
    'wind_speed': ['mean', 'max']
}

daily_era = era.groupby(['latitude','longitude','date']).agg(agg)
daily_era.columns = ['_'.join(col) for col in daily_era.columns]
daily_era = daily_era.reset_index()

daily_era.rename(columns={
    'precipitation_sum':'P_0d',
    'precipitation_max':'P_max1h_0d',
    'surface_runoff_sum':'R_0d',
    'surface_runoff_mean':'R_mean_0d',
    'temperature_mean':'T_mean_0d',
    'temperature_max':'T_max_0d',
    'wind_speed_mean':'WS_mean_0d',
    'wind_speed_max':'WS_max_0d'
}, inplace=True)

daily_era.head()


,latitude,longitude,date,P_0d,P_max1h_0d,R_0d,R_mean_0d,T_mean_0d,T_max_0d,WS_mean_0d,WS_max_0d
0,26.5,88.5,1979-01-01,0.0,0.0,0.0,0.0,16.240891,23.847076,1.115465,1.357756
1,26.5,88.5,1979-01-02,0.0,0.0,0.0,0.0,15.712799,23.569489,1.102641,1.364766
2,26.5,88.5,1979-01-03,0.0,0.0,0.0,0.0,15.354050,23.505035,1.139034,1.391488
3,26.5,88.5,1979-01-04,0.0,0.0,0.0,0.0,15.968239,23.729645,1.213820,1.545479
4,26.5,88.5,1979-01-05,0.0,0.0,0.0,0.0,15.757431,23.017242,1.351615,1.689649


### Map Floods to Nearest ERA5 Grid Cell

In [51]:
# === Build ERA5 grid from the daily table ===
grid = (
    daily_era[['latitude', 'longitude']]
    .drop_duplicates()
    .reset_index(drop=True)
)
grid_np = grid[['latitude', 'longitude']].to_numpy()

# --- Haversine helpers ---
import numpy as np

R_EARTH_KM = 6371.0

def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two points (deg)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return 2 * R_EARTH_KM * np.arcsin(np.sqrt(a))

def nearest_grid_idx_and_dist(lat, lon):
    """Return (index, distance_km) of nearest ERA5 grid point to (lat, lon)."""
    # quick approximate search via squared diff (good enough to pick the candidate)
    diffs = grid_np - np.array([lat, lon])
    idx = np.argmin((diffs**2).sum(axis=1))
    # precise great-circle distance for that candidate
    d_km = haversine_km(lat, lon, grid_np[idx, 0], grid_np[idx, 1])
    return idx, d_km

# === Map each flood record to a grid point ===
mapped = flood_df[['Date', 'District', 'Location', 'Latitude', 'Longitude']].copy()
mapped['date'] = pd.to_datetime(mapped['Date']).dt.floor('D')  # daily key (tz-naive local)

idxs, dists = [], []
for lat, lon in mapped[['Latitude', 'Longitude']].to_numpy():
    i, d = nearest_grid_idx_and_dist(lat, lon)
    idxs.append(i); dists.append(d)

mapped['grid_index']  = idxs
mapped['distance_km'] = dists

# attach grid lat/lon columns
mapped = (
    mapped.merge(grid.assign(grid_index=grid.index), on='grid_index', how='left')
          .rename(columns={'latitude': 'grid_lat', 'longitude': 'grid_lon'})
)

# optional: sanity check
print(mapped[['Date','District','Location','Latitude','Longitude','grid_lat','grid_lon','distance_km']].head())


        Date         District       Location  Latitude  Longitude  grid_lat  \
0 1980-07-10  Wangduephodrang   Wangdue town   27.4867    89.8995     27.50   
1 1990-08-08          Sarpang        Sarpang   26.8627    90.2717     26.75   
2 1994-07-10          Punakha        Punakha   27.5916    89.8774     27.50   
3 1996-08-14    Phuentsholing  Phuentsholing   26.8635    89.3883     26.75   
4 2000-07-16    Phuentsholing  Phuentsholing   26.8635    89.3883     26.75   

   grid_lon  distance_km  
0     90.00    10.022733  
1     90.25    12.715377  
2     90.00    15.806402  
3     89.50    16.797986  
4     89.50    16.797986  


In [52]:

events = mapped.drop_duplicates(subset=['grid_lat','grid_lon','date']).copy()
events['flood'] = 1
events.shape, events.head()


((105, 11),
         Date         District       Location  Latitude  Longitude       date  \
 0 1980-07-10  Wangduephodrang   Wangdue town   27.4867    89.8995 1980-07-10   
 1 1990-08-08          Sarpang        Sarpang   26.8627    90.2717 1990-08-08   
 2 1994-07-10          Punakha        Punakha   27.5916    89.8774 1994-07-10   
 3 1996-08-14    Phuentsholing  Phuentsholing   26.8635    89.3883 1996-08-14   
 4 2000-07-16    Phuentsholing  Phuentsholing   26.8635    89.3883 2000-07-16   
 
    grid_index  distance_km  grid_lat  grid_lon  flood  
 0          66    10.022733     27.50     90.00      1  
 1          22    12.715377     26.75     90.25      1  
 2          66    15.806402     27.50     90.00      1  
 3          19    16.797986     26.75     89.50      1  
 4          19    16.797986     26.75     89.50      1  )

In [53]:

daily_era['year'] = daily_era['date'].dt.year
daily_era['month'] = daily_era['date'].dt.month

event_keys = set(zip(events['grid_lat'], events['grid_lon'], events['date']))
event_df = events[['grid_lat','grid_lon','date']].copy()
event_df['year'] = event_df['date'].dt.year
event_df['month'] = event_df['date'].dt.month

candidates = daily_era.groupby(['grid_lat','grid_lon','month'], as_index=False) \
    .apply(lambda g: g['date'].tolist()).rename(columns={0:'dates'})

K = 3
ctrl_rows = []
for lat, lon, dt, yr, mo in event_df[['grid_lat','grid_lon','date','year','month']].itertuples(index=False, name=None):
    cand_list = candidates.loc[
        (candidates['grid_lat']==lat) &
        (candidates['grid_lon']==lon) &
        (candidates['month']==mo), 'dates']
    if len(cand_list)==0:
        continue
    pool = [d for d in cand_list.iloc[0] if (d.year != yr) and ((lat,lon,d) not in event_keys)]
    if len(pool)==0:
        continue
    take = np.random.choice(pool, size=min(K, len(pool)), replace=False)
    for d in take:
        ctrl_rows.append((lat, lon, d, 0))

controls = pd.DataFrame(ctrl_rows, columns=['grid_lat','grid_lon','date','flood'])
controls.shape, controls.head()


KeyError: 'grid_lat'